In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv('./digit-recognizer/train.csv')
data.head()

In [ ]:
data = np.array(data)
m, n = data.shape
np.random.shuffle(data)
data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n]

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n]

In [ ]:
def ReLU(z):
    return np.maximum(0, z)

def derivative_ReLU(z):
    return z > 0

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1 
    return one_hot_Y.T

In [ ]:
def init_params():
    w1 = np.random.rand(10, 784)
    b1 = np.random.rand(10, 1)
    w2 = np.random.rand(10, 10)
    b2 = np.random.rand(10, 1)

    return w1, w2, b1, b2 

In [ ]:
def Forward_prop(w1, b1, w2, b2, X):
    z1 = w1.dot(X) + b1
    a1 = ReLU(z1)
    z2 = w2.dot(a1) + b2
    a2 = softmax(z2)
    return z1, z2, a1, a2

def Backprop(z1, z2, a1, a2, w2, X, Y):
    m = Y.size 
    one_hot_Y = one_hot(Y)
    delta_z2 = a2 - one_hot_Y
    delta_w2 = 1 / m * delta_z2.dot(a1.T) 
    delta_b2 = 1 / m * np.sum(delta_z2, axis=1, keepdims=True) 
    
    delta_z1 = w2.T.dot(delta_z2) * derivative_ReLU(z1)
    delta_w1 = 1 / m * delta_z1.dot(X.T)
    delta_b1 = 1 / m * np.sum(delta_z1, axis=1, keepdims=True)
    return delta_w1, delta_w2, delta_b1, delta_b2


### why same alpha for every update??

In [ ]:
def update_params(w1, w2, b1, b2, delta_w1, delta_w2, delta_b1, delta_b2, alpha):
    w1 = w1 - alpha * delta_w1
    w2 = w2 - alpha * delta_w2
    b1 = b1 - alpha * delta_b1
    b2 = b2 - alpha * delta_b2

    return w1, w2, b1, b2

In [ ]:
def get_predictions(a2):
    return np.argmax(a2, 0)

def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / Y.size


def gradient_descent(X, Y, iterations, alpha):
    w1, w2, b1, b2 = init_params()

    for i in range(iterations):
        z1, z2, a1, a2 = Forward_prop(w1, b1, w2, b2, X)
        delta_w1, delta_w2, delta_b1, delta_b2 = Backprop(z1, z2, a1, a2, w2, X, Y)
        w1, w2, b1, b2 = update_params(w1, w2, b1, b2, delta_w1, delta_w2, delta_b1, delta_b2, alpha)

        if i%10 == 0:
            print("Iteration ", i)
            print("Accuracy: ", get_accuracy(get_predictions(a2), Y))

In [ ]:
w1, w2, b1, b2 = gradient_descent(X_train, Y_train, 1000, 0.1)